# Measuring the Lift [Step 4 - Precision@k Before and After]

> **MLCourse - Agentic AI - Advanced RAG - Reranking**

So far we have looked at rankings and said "that looks better". This notebook
replaces the eyeballing with a number.

We measure **precision@k** for four retrieval configurations over a fixed set of
questions, and report the real delta - **including if it turns out to be small**.
Honest measurement is the point of this notebook. A technique that helps by two
percentage points on your corpus is a very different engineering decision from
one that helps by twenty, and the only way to tell them apart is to measure on
*your* data.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


In [4]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(p) for p in paragraphs])

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)

print("BM25 documents :", len(paragraphs))
print("dense vectors  :", doc_vectors.shape)


def bm25_rank(query, top_n=50):
    """Return document indices ranked by BM25 score (best first)."""
    scores = bm25.get_scores(tokenize(query))
    return list(np.argsort(scores)[::-1][:top_n])


def dense_rank(query, top_n=50):
    """Return document indices ranked by cosine similarity (best first)."""
    qv = encoder.encode([query], normalize_embeddings=True)[0]
    sims = doc_vectors @ qv
    return list(np.argsort(sims)[::-1][:top_n])


def rrf(rankings, k=60, top_n=50):
    """Reciprocal Rank Fusion - the exact algorithm taught in
    ../01_hybrid_search/03_reciprocal_rank_fusion.ipynb."""
    scores = {}
    for ranked in rankings:
        for rank, doc_id in enumerate(ranked, 1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [int(doc_id) for doc_id, _ in ordered[:top_n]]


def hybrid_rank(query, top_n=50):
    """BM25 + dense, fused with RRF. This is our first-stage retriever."""
    return rrf([bm25_rank(query, top_n), dense_rank(query, top_n)], top_n=top_n)


print("hybrid top-3 for 'the queen and the croquet game':")
for i, doc_id in enumerate(hybrid_rank("the queen and the croquet game", 3), 1):
    print(f"  {i}. doc_{doc_id}: {paragraphs[doc_id][:90]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BM25 documents : 237
dense vectors  : (237, 384)
hybrid top-3 for 'the queen and the croquet game':
  1. doc_150: “Get to your places!” shouted the Queen in a voice of thunder, and people began running ab...
  2. doc_105: The Fish-Footman began by producing from under his arm a great letter, nearly as large as ...
  3. doc_152: The players all played at once without waiting for turns, quarrelling all the while, and f...


### 2. Ground truth without hand-waving

Any retrieval metric needs a definition of "relevant". Three options exist:

1. **Human labels** - the gold standard, expensive, does not scale.
2. **LLM-as-judge** - scalable, but the judge is noisy and costs tokens; this is
   what [`../10_rag_evaluation`](../10_rag_evaluation/README.md) explores.
3. **Rule-based labels** - cheap, perfectly reproducible, and honest as long as
   you write the rules *before* seeing the results.

We use option 3, because for a comparison between two retrievers what matters
most is that the judgement is **identical and unbiased across both**. A
paragraph is relevant to a question if it contains every keyword in at least one
of that question's keyword groups.

Writing these rules first, then running the experiment once, is what keeps the
result trustworthy.

In [5]:
# An evaluation question is only useful if we can decide, mechanically and
# without an LLM, whether a retrieved paragraph is relevant. We do that with
# required keyword sets: a paragraph counts as relevant when it contains every
# keyword in at least one of the "any_of" groups. This is a strict, honest,
# reproducible judgement - no LLM grading, no hand-waving.

EVAL_QUESTIONS = [
    {"q": "Why was the White Rabbit in such a hurry?",
     "any_of": [["rabbit", "hurry"], ["rabbit", "late"], ["oh dear", "late"]]},
    {"q": "What happened when Alice drank from the little bottle?",
     "any_of": [["drink", "bottle"], ["bottle", "shutting up like a telescope"],
                ["drank", "telescope"]]},
    {"q": "What game does the Queen of Hearts make everyone play?",
     "any_of": [["croquet"], ["flamingo", "hedgehog"]]},
    {"q": "Who does Alice meet at the mad tea party?",
     "any_of": [["hatter", "dormouse"], ["march hare", "hatter"], ["tea", "dormouse"]]},
    {"q": "What advice does the Caterpillar give Alice?",
     "any_of": [["caterpillar", "mushroom"], ["caterpillar", "keep your temper"],
                ["caterpillar", "who are you"]]},
    {"q": "How does the Cheshire Cat disappear?",
     "any_of": [["grin", "vanish"], ["cheshire cat", "grin"], ["vanished", "grin"]]},
    {"q": "What does the Queen shout whenever she is angry?",
     "any_of": [["off with"], ["queen", "executed"]]},
    {"q": "What happens at the trial of the Knave of Hearts?",
     "any_of": [["knave", "tarts"], ["jury", "verdict"], ["sentence", "verdict"]]},
]


def is_relevant(doc_text, question):
    """True when the paragraph satisfies any one keyword group for the question."""
    low = doc_text.lower()
    return any(all(word in low for word in group) for group in question["any_of"])


def precision_at_k(ranked_ids, question, k=5):
    """Fraction of the top-k retrieved paragraphs that are relevant."""
    top = ranked_ids[:k]
    return sum(is_relevant(paragraphs[i], question) for i in top) / max(len(top), 1)


# Sanity check: every question must have at least one relevant paragraph in
# the corpus, otherwise the metric is meaningless.
for question in EVAL_QUESTIONS:
    n_rel = sum(is_relevant(p, question) for p in paragraphs)
    print(f"{n_rel:3d} relevant paragraphs | {question['q']}")

  7 relevant paragraphs | Why was the White Rabbit in such a hurry?
  3 relevant paragraphs | What happened when Alice drank from the little bottle?
  9 relevant paragraphs | What game does the Queen of Hearts make everyone play?
 10 relevant paragraphs | Who does Alice meet at the mad tea party?
  2 relevant paragraphs | What advice does the Caterpillar give Alice?
  1 relevant paragraphs | How does the Cheshire Cat disappear?
  5 relevant paragraphs | What does the Queen shout whenever she is angry?
  1 relevant paragraphs | What happens at the trial of the Knave of Hearts?


Every question has at least one relevant paragraph, so precision@k is
well-defined for all of them. Some have only one or two - those are the hard
questions, and they are where reranking has the most room to help.

### 3. The four configurations

| name | stage 1 | stage 2 |
|---|---|---|
| `dense` | embedding search, top k | - |
| `bm25` | keyword search, top k | - |
| `hybrid` | BM25 + dense fused with RRF, top k | - |
| `hybrid+rerank` | hybrid RRF, **top 50 candidates** | cross-encoder, keep top k |

The only difference between the last two rows is the reranker. That isolation is
what makes the delta meaningful.

In [6]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
N_CANDIDATES = 50


def rerank_ids(query, candidate_ids):
    scores = cross_encoder.predict(
        [(query, paragraphs[i]) for i in candidate_ids], batch_size=32
    )
    return [int(i) for i, _ in sorted(zip(candidate_ids, scores), key=lambda x: -x[1])]


CONFIGS = {
    "dense":         lambda q: dense_rank(q, N_CANDIDATES),
    "bm25":          lambda q: bm25_rank(q, N_CANDIDATES),
    "hybrid":        lambda q: hybrid_rank(q, N_CANDIDATES),
    "hybrid+rerank": lambda q: rerank_ids(q, hybrid_rank(q, N_CANDIDATES)),
}

print("configurations:", list(CONFIGS))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

configurations: ['dense', 'bm25', 'hybrid', 'hybrid+rerank']


### 4. Run the experiment

Note there are no LLM calls in this loop - retrieval evaluation is pure
retrieval. That makes it fast, free, and repeatable, which is exactly what you
want when tuning a retriever.

In [7]:
K_VALUES = [1, 3, 5, 10]

results = {name: {k: [] for k in K_VALUES} for name in CONFIGS}

for question in EVAL_QUESTIONS:
    for name, retrieve in CONFIGS.items():
        ranked = retrieve(question["q"])
        for k in K_VALUES:
            results[name][k].append(precision_at_k(ranked, question, k))

print("evaluated", len(EVAL_QUESTIONS), "questions x", len(CONFIGS), "configurations")

evaluated 8 questions x 4 configurations


In [8]:
import numpy as np

print(f"{'configuration':<16}" + "".join(f"  P@{k:<6}" for k in K_VALUES))
print("-" * 52)
means = {}
for name in CONFIGS:
    means[name] = {k: float(np.mean(results[name][k])) for k in K_VALUES}
    row = "".join(f"  {means[name][k]:<7.3f}" for k in K_VALUES)
    print(f"{name:<16}{row}")

configuration     P@1       P@3       P@5       P@10    
----------------------------------------------------
dense             0.500    0.333    0.250    0.200  
bm25              0.375    0.292    0.275    0.188  
hybrid            0.500    0.333    0.275    0.200  
hybrid+rerank     0.500    0.458    0.325    0.250  


In [9]:
print("LIFT from reranking (hybrid+rerank minus hybrid)\n")
print(f"{'k':>3}  {'hybrid':>8}  {'reranked':>9}  {'absolute':>9}  {'relative':>9}")
print("-" * 48)
for k in K_VALUES:
    base = means["hybrid"][k]
    new = means["hybrid+rerank"][k]
    rel = (new - base) / base * 100 if base else float("nan")
    print(f"{k:>3}  {base:>8.3f}  {new:>9.3f}  {new - base:>+9.3f}  {rel:>+8.1f}%")

LIFT from reranking (hybrid+rerank minus hybrid)

  k    hybrid   reranked   absolute   relative
------------------------------------------------
  1     0.500      0.500     +0.000      +0.0%
  3     0.333      0.458     +0.125     +37.5%
  5     0.275      0.325     +0.050     +18.2%
 10     0.200      0.250     +0.050     +25.0%


### 5. Per-question detail - where the average comes from

An average hides everything interesting. Reranking usually helps a lot on a few
questions, does nothing on most, and occasionally hurts one. Look at the shape,
not just the mean.

In [10]:
K = 5
print(f"per-question precision@{K}\n")
print(f"{'question':<46} {'hybrid':>7} {'rerank':>7} {'delta':>7}")
print("-" * 70)

wins = losses = ties = 0
for i, question in enumerate(EVAL_QUESTIONS):
    base = results["hybrid"][K][i]
    new = results["hybrid+rerank"][K][i]
    delta = new - base
    if delta > 1e-9:
        wins += 1
    elif delta < -1e-9:
        losses += 1
    else:
        ties += 1
    print(f"{question['q'][:44]:<46} {base:>7.2f} {new:>7.2f} {delta:>+7.2f}")

print("-" * 70)
print(f"improved: {wins}   unchanged: {ties}   regressed: {losses}")

per-question precision@5

question                                        hybrid  rerank   delta
----------------------------------------------------------------------
Why was the White Rabbit in such a hurry?         0.40    0.60   +0.20
What happened when Alice drank from the litt      0.40    0.40   +0.00
What game does the Queen of Hearts make ever      0.20    0.20   +0.00
Who does Alice meet at the mad tea party?         0.40    0.40   +0.00
What advice does the Caterpillar give Alice?      0.40    0.20   -0.20
How does the Cheshire Cat disappear?              0.20    0.20   +0.00
What does the Queen shout whenever she is an      0.00    0.40   +0.40
What happens at the trial of the Knave of He      0.20    0.20   +0.00
----------------------------------------------------------------------
improved: 2   unchanged: 5   regressed: 1


### 6. MRR - a metric that cares about position 1

Precision@k treats all k slots equally: a relevant document at rank 5 counts the
same as one at rank 1. For RAG that is not quite right, because LLMs attend more
strongly to what appears early in the context.

**Mean Reciprocal Rank** fixes that. For each query take `1 / rank_of_first_relevant_document`
and average. A first-position hit scores 1.0; rank 5 scores 0.2. Rerankers
typically look better under MRR than under precision@k, because promoting the
single best document to position 1 is precisely what they do.

In [11]:
def reciprocal_rank(ranked_ids, question, limit=20):
    for pos, doc_id in enumerate(ranked_ids[:limit], 1):
        if is_relevant(paragraphs[doc_id], question):
            return 1.0 / pos
    return 0.0


print(f"{'configuration':<16} {'MRR':>8}")
print("-" * 26)
mrr = {}
for name, retrieve in CONFIGS.items():
    mrr[name] = float(np.mean([reciprocal_rank(retrieve(q["q"]), q)
                               for q in EVAL_QUESTIONS]))
    print(f"{name:<16} {mrr[name]:>8.3f}")

print(f"\nMRR lift from reranking: {mrr['hybrid+rerank'] - mrr['hybrid']:+.3f} "
      f"({(mrr['hybrid+rerank'] - mrr['hybrid']) / mrr['hybrid'] * 100:+.1f}%)")

configuration         MRR
--------------------------
dense               0.681
bm25                0.567
hybrid              0.682


hybrid+rerank       0.698

MRR lift from reranking: +0.016 (+2.3%)


### 7. Have the LLM read the scoreboard

A nice use of the model here is turning the table into an engineering
recommendation. This is not the model measuring anything - the numbers are
already fixed - it is summarising them.

In [12]:
summary_table = "\n".join(
    f"{name}: " + ", ".join(f"P@{k}={means[name][k]:.3f}" for k in K_VALUES)
    + f", MRR={mrr[name]:.3f}"
    for name in CONFIGS
)

verdict = ask(
    "You are a retrieval engineer. Below are measured retrieval metrics on a "
    "237-paragraph corpus with 8 evaluation questions, comparing four "
    "configurations. State in 4-6 sentences: which configuration wins, how large "
    "the reranking lift actually is, and whether that lift justifies roughly "
    "200ms of extra query latency. Be blunt if the lift is small.\n\n"
    + summary_table
)
print(summary_table)
print("\n--- model's read of the numbers ---")
print(verdict)

dense: P@1=0.500, P@3=0.333, P@5=0.250, P@10=0.200, MRR=0.681
bm25: P@1=0.375, P@3=0.292, P@5=0.275, P@10=0.188, MRR=0.567
hybrid: P@1=0.500, P@3=0.333, P@5=0.275, P@10=0.200, MRR=0.682
hybrid+rerank: P@1=0.500, P@3=0.458, P@5=0.325, P@10=0.250, MRR=0.698

--- model's read of the numbers ---
The hybrid+rerank configuration is the clear winner, achieving the highest scores across all metrics, particularly with a P@3 of 0.458 and MRR of 0.698. However, the reranking lift is modest, adding only 0.016 to MRR and 0.125 to P@3 compared to the base hybrid model. This marginal improvement suggests that the reranker is not fundamentally fixing retrieval errors but rather fine-tuning the order of already-relevant candidates. Given that the base hybrid model already matches the dense model's top-1 performance and outperforms BM25 significantly, the additional complexity is hard to justify. Unless the application is extremely sensitive to the top-3 ranking precision, the 200ms latency penalty is l

### 8. Reading your own results honestly

Some things to hold in mind when the number is smaller than the blog posts
promised:

- **A small corpus caps the lift.** With 237 paragraphs, dense search alone
  already has the right document in the top 10 most of the time. Reranking earns
  its keep when the candidate pool is noisy - large corpora, many near-duplicates,
  ambiguous queries.
- **Keyword-based ground truth flatters lexical retrievers.** BM25 is judged by
  a rule that rewards the very keywords it matches on. A cross-encoder that
  finds a *paraphrase* of the answer gets no credit under our rule. Real lift is
  likely understated here.
- **Eight questions is a small sample.** With 8 queries, one question changing
  moves the mean by 0.125 at k=1. Do not over-read a small delta - build a
  bigger question set before making a production decision.
- **MRR usually moves more than P@k.** Getting the best document into slot 1 is
  the reranker's speciality, and it is also what most improves generation.

The discipline that matters: report what you measured, state the caveats, and
size your evaluation set to the size of the decision.

### 9. Key takeaways

- Define relevance **before** running the experiment, and apply the identical
  rule to every configuration.
- Precision@k and MRR answer different questions; report both.
- Look at per-question wins/losses, not only the mean.
- Expect modest lifts on small clean corpora and larger lifts on big noisy ones.
- If the measured lift does not justify the latency on your data, **do not ship
  the reranker** - which is what the next notebook helps you price.

Next: [`05_latency_and_cost_tradeoff.ipynb`](05_latency_and_cost_tradeoff.ipynb).